# 03 — Spatial Exploration: Production & Mesoregions

**Objective**: Map the spatial distribution of arabica coffee production across Brazil's municipalities and mesoregions.

**Data**: IBGE Municipal Agricultural Survey 2018 — 1,227 producing municipalities.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.graph_objects as go
import plotly.express as px

from src.config import DATA_RAW_DIR, MESOREGIONS
from src.visualization.production_maps import (
    municipality_choropleth,
    mesoregion_bar_chart,
    production_by_state_pie,
)

gdf = gpd.read_file(DATA_RAW_DIR / "coffee_producing_municipalities.geojson")
print(f"Loaded {len(gdf)} producing municipalities")
print(f"Columns: {list(gdf.columns)}")
gdf.head()

## 3.1 Interactive Production Choropleth

Each municipality colored by its 2018 arabica production. Darker = more tonnes.

In [ ]:
municipality_choropleth().show()

## 3.2 Production by State

In [ ]:
production_by_state_pie().show()

## 3.3 Top Producing Mesoregions

In [ ]:
mesoregion_bar_chart(20).show()

## 3.4 Production Concentration Analysis

What share of arabica production comes from the top N municipalities?

In [ ]:
total = gdf["production_of_arabica_coffee_tonnes_2018"].sum()
sorted_prod = gdf.sort_values("production_of_arabica_coffee_tonnes_2018", ascending=False)
sorted_prod["cumsum_pct"] = sorted_prod["production_of_arabica_coffee_tonnes_2018"].cumsum() / total * 100

concentration = pd.DataFrame({
    "Top N": [1, 5, 10, 20, 50, 100, 200, 500],
    "Cumulative %": [
        sorted_prod.iloc[:n]["production_of_arabica_coffee_tonnes_2018"].sum() / total * 100
        for n in [1, 5, 10, 20, 50, 100, 200, 500]
    ],
})
concentration

## 3.5 Municipality → Mesoregion Mapping

Verify the lookup between each municipality and its parent mesoregion.

In [ ]:
gdf["meso_key"] = (
    gdf["state_geocode"].astype(str).str.zfill(2)
    + "_"
    + gdf["meso_id"].astype(str).str.zfill(2)
)

meso_summary = gdf.groupby(["state_name", "meso_name"]).agg(
    municipalities=("municipality_name", "count"),
    total_tonnes=("production_of_arabica_coffee_tonnes_2018", "sum"),
).sort_values("total_tonnes", ascending=False).reset_index()

meso_summary["share_pct"] = (meso_summary["total_tonnes"] / total * 100).round(1)
meso_summary.head(15)

## Key Observations

- Fill in your observations here after running the cells above.